In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 

words = open('names.txt','r').read().split()

len(words)

32033

In [2]:
# Building vocab of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [3]:
# Building the dataset
def build_dataset(words):
    block_size = 3 # the context length: how many characters do we take to predict the next one
    X, Y = [],[]

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            idx = stoi[ch]
            X.append(context)
            Y.append(idx)
            context = context[1:] + [idx] 

    # The dataset
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

trian_num = int(0.8 * len(words))
test_num = int(.9*len(words))

Xtr, Ytr = build_dataset(words[:trian_num])
Xval, Yval = build_dataset(words[trian_num:test_num])
Xtes, Ytes = build_dataset(words[:test_num])

In [4]:
g = torch.Generator().manual_seed(2147483647)
# Embedding creation
C = torch.randn((27,15), generator=g) # initalizes random parameters 

In [ ]:
# Creating random weights and bias
W1 = torch.randn((45,300), generator=g) # 15 inputs (from embeddings) and 100 neurons 
b1 = torch.randn(300,generator=g)
W2 = torch.randn(300,27,generator=g) # output parameters
b2 = torch.randn(27,generator=g)
params = [C,W1,b1,W2,b2]

for p in params:
    p.requires_grad = True

In [6]:
sum(p.nelement() for p in params)

22332

In [7]:
# Finding optimal learning rate 
lre = torch.linspace(-3,0,1000)
lrs = 10** lre

lrs

tensor([0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0011,
        0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011, 0.0011,
        0.0011, 0.0011, 0.0011, 0.0012, 0.0012, 0.0012, 0.0012, 0.0012, 0.0012,
        0.0012, 0.0012, 0.0012, 0.0012, 0.0012, 0.0012, 0.0013, 0.0013, 0.0013,
        0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0013, 0.0014,
        0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014, 0.0014,
        0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015, 0.0015,
        0.0015, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016,
        0.0016, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017, 0.0017,
        0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0018, 0.0019,
        0.0019, 0.0019, 0.0019, 0.0019, 0.0019, 0.0019, 0.0019, 0.0020, 0.0020,
        0.0020, 0.0020, 0.0020, 0.0020, 0.0020, 0.0021, 0.0021, 0.0021, 0.0021,
        0.0021, 0.0021, 0.0021, 0.0022, 

In [14]:
lri = []
lossi = []

# ---- Training Loop ------
for i in range(250000):
    # --- mini batch ---
    idx = torch.randint(0,Xtr.shape[0], (64,))

    # ---- Forward Pass ---
    # embedding look up
    emb = C[Xtr[idx]] 
    # Constrcuting hidden layer and activating it - first flattening the embedded inputs
    hidden = torch.tanh((emb.view(-1, 45) @ W1) + b1)

    # output layer 
    logits = (hidden @ W2) + b2
    loss = F.cross_entropy(logits,Ytr[idx])
    
    # ---- Backward Pass ---
    for p in params:
        p.grad = None

    loss.backward()

    # ---- Update ---
    lr = .001
    for p in params:
        p.data += (-lr * p.grad)

    # -- track stats ---
    #lri.append(lre[i])
    #lossi.append(loss.item())


In [15]:
# plt.plot(lri,lossi)

In [16]:
# Loss on whole network
emb = C[Xtr]
hidden = torch.tanh((emb.view(-1, 45) @ W1) + b1)
logits = (hidden @ W2) + b2
loss = F.cross_entropy(logits,Ytr)

loss.item() 

1.9815332889556885

In [17]:
# Loss on dev/val 
emb = C[Xval]
hidden = torch.tanh((emb.view(-1, 45) @ W1) + b1)
logits = (hidden @ W2) + b2
loss = F.cross_entropy(logits,Yval)

loss.item() 

2.411705255508423

In [21]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
  out = []
  context = [0] * 3
  while True:
    emb = C[torch.tensor([context])]          # (1, 3, 15)
    h = torch.tanh(emb.view(1, -1) @ W1 + b1) # (1, 300)
    logits = h @ W2 + b2                      # (1, 27)
    probs = F.softmax(logits, dim=1)
    idx = torch.multinomial(probs, num_samples=1, generator=g).item()
    context = context[1:] + [idx]
    out.append(idx)
    if idx == 0:
      break

  print(''.join(itos[i] for i in out))


mora.
kayah.
seel.
nihayla.
reisha.
endra.
grace.
korelin.
shi.
jenne.
elisson.
arleigh.
malkea.
noshubhrushiriel.
kinleen.
jerenteron.
urented.
ryyah.
faeh.
yuma.
